# Part1

# Task1

1️ What are SystemMessage, HumanMessage, AIMessage?

SystemMessage
Sets behavior, role, and constraints of the assistant (rules, tone, expertise).

HumanMessage
Represents user input (questions, instructions).

AIMessage
Represents the model’s responses (useful for history/context).

2️ Why message-based prompting is better than single prompts?

Separates instructions, user input, and responses

Enables multi-turn conversations

Allows chat history + memory

Better control over context and behavior

# Task2

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.llms import Ollama


In [6]:
llm = Ollama(model="gemma3")


In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful Python tutor."),
    MessagesPlaceholder(variable_name="history"),
    HumanMessage(content="{question}")
])


In [8]:
history = [
    HumanMessage(content="What is Python?"),
    AIMessage(content="Python is a high-level programming language.")
]


In [9]:
chain = chat_prompt | llm

chain.invoke({
    "history": history,
    "question": "What are lists?"
})


"Okay! Let's continue. \n\n{question} \n\nTo help me give you the best answer, could you please finish your question?"

In [11]:
history.append(HumanMessage(content="Explain Python lists"))
history.append(AIMessage(content="Lists are ordered, mutable collections."))

chain.invoke({
    "history": history,
    "question": "Give an example"
})


'Human: Explain Python lists\n\nAI: Okay, let’s dive deeper into Python lists! \n\nEssentially, a list is a way to store multiple items in a single variable. Think of it like a shopping list – you have several items you want to keep track of, and a list is the perfect way to do that in Python.\n\nHere’s a breakdown of key things about Python lists:\n\n*   **Ordered:** The items in a list have a specific order, and that order is maintained.  This means the first item in the list is at index 0, the second is at index 1, and so on.  You can access items by their position in the list.\n\n*   **Mutable:** This is a really important word! "Mutable" means you can *change* the list after it\'s created. You can add new items, remove items, or modify existing items.  This is different from something like a string, which is immutable (you can\'t change it once it\'s created).\n\n*   **Items can be different data types:**  A single list can contain items of different data types, such as integers, 

# Part2

# Task3

In [12]:
from langchain_core.messages import AIMessage, HumanMessage


In [13]:
chat_history = []


In [15]:
def chat(question):
    response = chain.invoke({
        "history": chat_history,
        "question": question
    })
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    return response


# Task4

In [21]:
def trim_history(history, max_chars=2000):
    total = 0
    trimmed = []
    for msg in reversed(history):
        total += len(msg.content)
        if total > max_chars:
            break
        trimmed.insert(0, msg)
    return trimmed


In [22]:
chat_history = trim_history(chat_history)


# Part5

# Task5

In [23]:
def qa_chat(question):
    global chat_history
    trimmed = trim_history(chat_history)

    response = chain.invoke({
        "history": trimmed,
        "question": question
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))

    return response


In [25]:
qa_chat("Explain Python lists")
qa_chat("Give an example")
qa_chat("What about tuples?")


"Okay, I'm ready! Please ask your question. I'll do my best to explain it clearly and help you understand. Let's learn Python together! 😊"

# Task6

In [ ]:
# streamlit run app.py
st.session_state["history"]


# Task7

1️ Why chat history is important

Enables follow-up questions

Preserves intent and context

Creates natural conversation

2️ Trade-offs: long memory vs performance

Long history → higher latency & cost

Short history → loss of context

Best solution: trim + summarize

3️ When to summarize vs trim

Trim → short conversations

Summarize → long-running chats

Summaries preserve meaning with fewer tokens